## Structured Output with Claude Models

### Installing Utilities and Libraries

In [ ]:
%pip install anthropic==0.120.2 python-dotenv==1.2.2 pydantic==2.13.4

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

### Create the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key = claude_api_key)

### Create the User Prompt

In [ ]:
user_prompt = """
Extract the following customer inquiry into structured JSON.

Customer Name: Sarah Johnson
Email: sarah.johnson@contoso.com

Company: Contoso Electronics

The customer is interested in our Premium Digital Marketing Package.

She would like to schedule a consultation next Wednesday and has requested additional pricing information.
"""

### Generate Structured Output using raw JSON Schema Configuration

In [ ]:
message = client.messages.create(
    model=claude_model_name,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "customer_name": {
                        "type": "string"
                    },
                    "email": {
                        "type": "string"
                    },
                    "company": {
                        "type": "string"
                    },
                    "service_requested": {
                        "type": "string"
                    },
                    "consultation_requested": {
                        "type": "boolean"
                    },
                    "pricing_information_requested": {
                        "type": "boolean"
                    }
                },
                "required": [
                    "customer_name",
                    "email",
                    "company",
                    "service_requested",
                    "consultation_requested",
                    "pricing_information_requested"
                ],
                "additionalProperties": False
            }
        }
    }
)

for block in message.content:
    if block.type == "text":
        print(block.text)

### Use Pydantic Models to Generate Structured Output

In [ ]:
from pydantic import BaseModel 

class CustomerLead(BaseModel):
    customer_name: str
    email: str
    company: str
    service_requested: str
    consultation_requested: bool
    pricing_information_requested: bool

In [ ]:
response = client.messages.parse(
    model=claude_model_name,
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    output_format = CustomerLead
)

contact = response.parsed_output
print("customer name: ", contact.customer_name)
print("email: ", contact.email)
print("company: ", contact.company)
print("service requested: ", contact.service_requested)
print("consultation requested: ", contact.consultation_requested)
print("pricing information requested: ", contact.pricing_information_requested)